In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!ls

drive  sample_data


In [3]:
%cd /content/drive/MyDrive/Intent-Classification-ML-Project/

/content/drive/MyDrive/Intent-Classification-ML-Project


Loading Dataset we created from RULE BASELINE

In [4]:
import pandas as pd

# Paths should match what you used in 01
features_path = "data/rule_baseline__v1+v2.parquet"

df = pd.read_parquet(features_path)

print("Loaded shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())  # first 40 cols, just to confirm

df.head()

Loaded shape: (300000, 70)

Columns:
['Login Timestamp', 'User ID', 'Round-Trip Time [ms]', 'IP Address', 'Country', 'Region', 'City', 'ASN', 'User Agent String', 'Browser Name and Version', 'OS Name and Version', 'Device Type', 'Login Successful', 'Is Attack IP', 'Is Account Takeover', 'browser', 'os', 'hour', 'dayofweek', 'is_new_device_for_user', 'is_new_ip_for_user', 'is_off_hours', 'failed_login', 'failures_last_5', 'failure_streak', 'failure_streak_capped', 'location', 'new_location_flag', 'new_asn_flag', 'device_fingerprint', 'valid_device', 'new_device_flag', 'device_change_rate', 'ts_sec', 'delta_sec', 'new_window', 'window_id', 'logins_5min', 'failure_flag', 'burst_failure_count', 'failure_rate', 'streak_reset', 'streak_id', 'failure_streak_length', 'location_freq', 'location_rarity', 'device_type_freq', 'device_type_rarity', 'asn_freq', 'asn_rarity', 'location_rarity_q', 'device_type_rarity_q', 'asn_rarity_q', 'user_offhour_rate', 'is_unusual_time_for_user', 'user_hour_std',

,Login Timestamp,User ID,Round-Trip Time [ms],IP Address,Country,Region,City,ASN,User Agent String,Browser Name and Version,...,rule_new_asn,rule_recent_failures,rule_risk_score,rule_risk_band,rule_risky_device_type,rule_high_risk_country,rule_risk_score_v2,rule_risk_band_v2,rule_decision_v2,pred_attack
0,2020-02-06 17:10:54.364,-9223287066183308537,541.0,84.209.76.159,no,oslo county,oslo,41164,Mozilla/5.0 (Macintosh; Intel Mac OS X 10_14_6...,Chrome 69.0.3497.17.19,...,0,0,0,low,0,0,0,low,ALLOW,0
1,2020-02-06 19:52:41.530,-9223258649185196422,541.0,91.208.148.129,no,-,-,49310,Mozilla/5.0 (iPad; CPU OS 7_1 like Mac OS X) ...,Android 2.3.3.2672,...,0,0,0,low,0,0,0,low,ALLOW,0
2,2020-02-06 20:55:19.627,-9223258649185196422,541.0,91.208.148.129,no,-,-,49310,Mozilla/5.0 (iPad; CPU OS 7_1 like Mac OS X) ...,Android 2.3.3.2672,...,0,0,0,low,0,0,0,low,ALLOW,0
3,2020-02-05 21:03:20.657,-9223200578825105501,541.0,79.161.56.83,no,vestfold og telemark,holmestrand,29695,Mozilla/5.0 (iPhone; CPU iPhone OS 13_4 like ...,Chrome Mobile 81.0.4044.2033,...,0,0,0,low,0,0,0,low,ALLOW,0
4,2020-02-06 19:12:29.501,-9223199305075633823,541.0,79.161.86.86,no,-,-,29695,Mozilla/5.0 (Linux; U; Android 13.0; i phone X...,Opera Mobile 52.1.2254,...,0,0,0,low,0,0,0,low,ALLOW,0


From DATASET:

Base / raw auth attributes (match Table II):

	•	Login Timestamp
	•	User ID
	•	IP Address
	•	Country, Region, City, ASN
	•	Device Type, OS Name and Version, Browser Name and Version, User Agent String
	•	Round-Trip Time [ms]
	•	Login Successful
	•	Is Attack IP
	•	Is Account Takeover
	•	plus simplified: browser, os, hour, dayofweek

These are the nodes & raw edges that KG “sees”.

Contextual/behavioral derived features (KG-style):

	•	Novelty / change:
	•	is_new_device_for_user
	•	is_new_ip_for_user
	•	new_location_flag
	•	new_asn_flag
	•	new_device_flag
	•	device_change_rate
	•	Temporal aggregates:
	•	failures_last_5
	•	failure_streak, failure_streak_capped, failure_streak_length
	•	logins_5min
	•	burst_failure_count
	•	failure_rate
	•	Time behavior:
	•	is_off_hours
	•	is_unusual_time_for_user
	•	user_offhour_rate
	•	user_hour_std, user_hour_std_q
	•	Global rarity / population stats:
	•	location_freq, location_rarity, location_rarity_q
	•	device_type_freq, device_type_rarity, device_type_rarity_q
	•	asn_freq, asn_rarity, asn_rarity_q

These are exactly what your paper calls “KG-derived contextual features” — they conceptually come from “relationships between user, device, location, ASN over time”.

Rule/policy signals (for policy layer):

	•	rule_unusual_time, rule_off_hours
	•	rule_new_device, rule_new_asn, rule_recent_failures
	•	rule_risky_device_type, rule_high_risk_country
	•	rule_risk_score, rule_risk_band
	•	rule_risk_score_v2, rule_risk_band_v2, rule_decision_v2
	•	pred_attack

These belong to your policy / rule-based side.

# We are going to use Location/ASN/DEVICE:


Because they are the strongest, most interpretable security signals you have in this dataset, and they give you maximum value for minimum complexity.

“Who is logging in, from which place, using which device, on which network, and how typical/risky is that compared to their own history and the population.”

# Creating canonical IDs for KG entities. To make it consistent for KG graph

In [5]:
import pandas as pd

df_kg = df.copy()

# Make sure timestamp is datetime and data is sorted per user + time
df_kg["Login Timestamp"] = pd.to_datetime(df_kg["Login Timestamp"])
df_kg = df_kg.sort_values(["User ID", "Login Timestamp"]).reset_index(drop=True)

# 1) location_id — reuse your existing "location" column
# (already something like "country,region,city" and normalized)
df_kg["location_id"] = df_kg["location"].fillna("unknown,unknown,unknown")

# 2) device_id — reuse your existing device_fingerprint
df_kg["device_id"] = df_kg["device_fingerprint"].fillna("unknown_device")

# 3) asn_id — from ASN
df_kg["asn_id"] = df_kg["ASN"].fillna("unknown_asn").astype(str)

df_kg[["User ID", "Login Timestamp", "location_id", "device_id", "asn_id"]].head()

,User ID,Login Timestamp,location_id,device_id,asn_id
0,-9223287066183308537,2020-02-06 17:10:54.364,"no,oslo county,oslo","desktop,Mac OS X 10.14.6,Chrome 69.0.3497.17.1...",41164
1,-9223258649185196422,2020-02-06 19:52:41.530,"no,-,-","mobile,iOS 7.1,Android 2.3.3.2672,Mozilla/5.0 ...",49310
2,-9223258649185196422,2020-02-06 20:55:19.627,"no,-,-","mobile,iOS 7.1,Android 2.3.3.2672,Mozilla/5.0 ...",49310
3,-9223200578825105501,2020-02-05 21:03:20.657,"no,vestfold og telemark,holmestrand","mobile,iOS 13.4,Chrome Mobile 81.0.4044.2033,M...",29695
4,-9223199305075633823,2020-02-06 19:12:29.501,"no,-,-","mobile,Android 13.0,Opera Mobile 52.1.2254,Moz...",29695


# Build KG edge tables

User–Location edges

In [6]:
user_location_edges = (
    df_kg
    .groupby(["User ID", "location_id"])
    .agg(
        events=("Login Timestamp", "count"),
        first_seen=("Login Timestamp", "min"),
        last_seen=("Login Timestamp", "max"),
    )
    .reset_index()
)

print("user_location_edges shape:", user_location_edges.shape)
user_location_edges.head()

user_location_edges shape: (127304, 5)


,User ID,location_id,events,first_seen,last_seen
0,-9223287066183308537,"no,oslo county,oslo",1,2020-02-06 17:10:54.364,2020-02-06 17:10:54.364
1,-9223258649185196422,"no,-,-",2,2020-02-06 19:52:41.530,2020-02-06 20:55:19.627
2,-9223200578825105501,"no,vestfold og telemark,holmestrand",1,2020-02-05 21:03:20.657,2020-02-05 21:03:20.657
3,-9223199305075633823,"no,-,-",1,2020-02-06 19:12:29.501,2020-02-06 19:12:29.501
4,-9223121694105191762,"no,oslo county,oslo",1,2020-02-06 09:30:49.497,2020-02-06 09:30:49.497


User–Device edges

In [7]:
user_device_edges = (
    df_kg
    .groupby(["User ID", "device_id"])
    .agg(
        events=("Login Timestamp", "count"),
        first_seen=("Login Timestamp", "min"),
        last_seen=("Login Timestamp", "max"),
    )
    .reset_index()
)

print("user_device_edges shape:", user_device_edges.shape)
user_device_edges.head()

user_device_edges shape: (140066, 5)


,User ID,device_id,events,first_seen,last_seen
0,-9223287066183308537,"desktop,Mac OS X 10.14.6,Chrome 69.0.3497.17.1...",1,2020-02-06 17:10:54.364,2020-02-06 17:10:54.364
1,-9223258649185196422,"mobile,iOS 7.1,Android 2.3.3.2672,Mozilla/5.0 ...",2,2020-02-06 19:52:41.530,2020-02-06 20:55:19.627
2,-9223200578825105501,"mobile,iOS 13.4,Chrome Mobile 81.0.4044.2033,M...",1,2020-02-05 21:03:20.657,2020-02-05 21:03:20.657
3,-9223199305075633823,"mobile,Android 13.0,Opera Mobile 52.1.2254,Moz...",1,2020-02-06 19:12:29.501,2020-02-06 19:12:29.501
4,-9223121694105191762,"desktop,Chrome OS 12105.100.0,Chrome 72.0.3626...",1,2020-02-06 09:30:49.497,2020-02-06 09:30:49.497


User–ASN edges

In [8]:
user_asn_edges = (
    df_kg
    .groupby(["User ID", "asn_id"])
    .agg(
        events=("Login Timestamp", "count"),
        first_seen=("Login Timestamp", "min"),
        last_seen=("Login Timestamp", "max"),
    )
    .reset_index()
)

print("user_asn_edges shape:", user_asn_edges.shape)
user_asn_edges.head()

user_asn_edges shape: (121840, 5)


,User ID,asn_id,events,first_seen,last_seen
0,-9223287066183308537,41164,1,2020-02-06 17:10:54.364,2020-02-06 17:10:54.364
1,-9223258649185196422,49310,2,2020-02-06 19:52:41.530,2020-02-06 20:55:19.627
2,-9223200578825105501,29695,1,2020-02-05 21:03:20.657,2020-02-05 21:03:20.657
3,-9223199305075633823,29695,1,2020-02-06 19:12:29.501,2020-02-06 19:12:29.501
4,-9223121694105191762,15659,1,2020-02-06 09:30:49.497,2020-02-06 09:30:49.497


Above 3 tables are your explicit KG views:

	•	Nodes: User ID, location_id, device_id, asn_id
 	Edges:
	•	user–location with how often and since when
	•	user–device with how often and since when
	•	user–ASN with how often and since when

  We Found:



```
	•	user_location_edges: 127k user↔location relationships
	•	user_device_edges: 140k user↔device relationships
	•	user_asn_edges: 121k user↔ASN relationships
```
user X has used ASN Y N times between first_seen and last_seen”



Explanation: Once you have these KG tables, we can:

	•	Add per-edge stats:
	•	success_rate per user–location
	•	attack_rate per user–device
	•	time-since-last-seen as a feature (recency)
	•	Add global stats (KG-level):
	•	how many users share this device/location/asn (user_count)
	•	how many attacks are associated with each node

#  Global KG stats (population-level view). Build global stats tables


This will define for each location_id or asn_id or device_id how many

  •	how many events
	•	how many unique users
	•	how many attacks
	•	attack rate

In [9]:
# We assume df_kg already exists, with:
# - Login Timestamp (datetime)
# - User ID
# - location_id
# - device_id
# - asn_id
# - Is Attack IP

# Make sure label is numeric
df_kg["Is Attack IP"] = df_kg["Is Attack IP"].astype(int)

# 3.1 Global stats for locations
location_global_stats = (
    df_kg
    .groupby("location_id")
    .agg(
        events=("Login Timestamp", "count"),
        users=("User ID", "nunique"),
        attacks=("Is Attack IP", "sum")
    )
    .reset_index()
)

location_global_stats["attack_rate"] = (
    location_global_stats["attacks"] / location_global_stats["events"]
)

print("location_global_stats shape:", location_global_stats.shape)
location_global_stats.head()

# 3.2 Global stats for devices
device_global_stats = (
    df_kg
    .groupby("device_id")
    .agg(
        events=("Login Timestamp", "count"),
        users=("User ID", "nunique"),
        attacks=("Is Attack IP", "sum")
    )
    .reset_index()
)

device_global_stats["attack_rate"] = (
    device_global_stats["attacks"] / device_global_stats["events"]
)

print("device_global_stats shape:", device_global_stats.shape)
device_global_stats.head()

# 3.3 Global stats for ASNs
asn_global_stats = (
    df_kg
    .groupby("asn_id")
    .agg(
        events=("Login Timestamp", "count"),
        users=("User ID", "nunique"),
        attacks=("Is Attack IP", "sum")
    )
    .reset_index()
)

asn_global_stats["attack_rate"] = (
    asn_global_stats["attacks"] / asn_global_stats["events"]
)

print("asn_global_stats shape:", asn_global_stats.shape)
asn_global_stats.head()

location_global_stats shape: (5841, 5)
device_global_stats shape: (20440, 5)
asn_global_stats shape: (2664, 5)


,asn_id,events,users,attacks,attack_rate
0,10085,6,3,0,0.0
1,10269,6,2,0,0.0
2,10778,44,11,0,0.0
3,10991,5,3,0,0.0
4,11114,2,1,0,0.0


We Found:

	•	location_global_stats: 5,841 unique locations
	•	device_global_stats: 20,440 unique device fingerprints
	•	asn_global_stats: 2,664 ASNs

For Report/Explanation. TOP RISKY ASN's

In [10]:
# Copy to avoid accidental mutation
asn_report = asn_global_stats.copy()

# Filter: at least 50 events and some attacks
asn_risky = asn_report[
    (asn_report["events"] >= 50) &
    (asn_report["attacks"] > 0)
].copy()

# Sort by attack_rate (descending) and maybe then by events
asn_risky = asn_risky.sort_values(
    by=["attack_rate", "events"],
    ascending=[False, False]
)

# Show top 10 for reporting
asn_risky.head(10)

,asn_id,events,users,attacks,attack_rate
1455,47655,86,1,86,1.000000
1557,500021,440,238,412,0.936364
2429,57423,197,111,184,0.934010
1574,500039,1423,27,1324,0.930429
1571,500035,138,16,107,0.775362
181,138039,58,6,41,0.706897
871,265355,62,32,43,0.693548
1263,398986,3029,774,2075,0.685045
564,206948,103,22,67,0.650485
703,22612,56,1,36,0.642857


TOP RISKY COUNTRY/LOCATION

In [11]:
loc_report = location_global_stats.copy()

# Filter: at least 200 events and non-zero attacks
loc_risky = loc_report[
    (loc_report["events"] >= 200) &
    (loc_report["attacks"] > 0)
].copy()

# Sort by attack_rate descending
loc_risky = loc_risky.sort_values(
    by=["attack_rate", "events"],
    ascending=[False, False]
)

# Top 10 risky locations
loc_risky.head(10)

,location_id,events,users,attacks,attack_rate
4683,"pl,silesia,częstochowa",337,74,324,0.961424
4905,"ro,bucuresti,bucharest",789,73,485,0.614702
5721,"us,texas,dallas",2400,608,1392,0.580000
4886,"ro,-,-",591,80,267,0.451777
5467,"us,-,-",53507,17860,17340,0.324070
5820,"vn,-,-",202,5,65,0.321782
2075,"id,bali,denpasar",663,131,156,0.235294
4676,"pl,silesia,bielsko-biala",429,86,93,0.216783
1512,"de,rheinland-pfalz,trier",241,47,34,0.141079
5130,"tr,-,-",1342,216,186,0.138599


Top risky devices

In [12]:
dev_report = device_global_stats.copy()

# Filter: at least 50 events, some attacks
dev_risky = dev_report[
    (dev_report["events"] >= 50) &
    (dev_report["attacks"] > 0)
].copy()

dev_risky = dev_risky.sort_values(
    by=["attack_rate", "events"],
    ascending=[False, False]
)

dev_risky.head(10)

,device_id,events,users,attacks,attack_rate
1216,"desktop,Mac OS X 11.6.3,Chrome 72.0.3626.115,M...",463,1,463,1.000000
4,"bot,Other ,Linkbot 1.0,ZoomBot (Linkbot 1.0 ht...",99,1,99,1.000000
18755,"mobile,iOS 14.2.1,Other ,nslookup -q=cname ob3...",123,1,122,0.991870
2066,"mobile,Android 12.0,Chrome Mobile WebView 30.0...",252,1,241,0.956349
1188,"desktop,Mac OS X 11.6.3,Chrome 64.0.3282,Mozil...",71,1,66,0.929577
4707,"mobile,Android 4.1,Chrome Mobile WebView 85.0....",56,1,42,0.750000
864,"desktop,Mac OS X 10.14.6,Chrome 72.0.3626.116,...",56,3,35,0.625000
6096,"mobile,Android 5.5.1,Chrome Mobile 81.0.4044.1...",94,32,51,0.542553
19200,"mobile,iOS 9.3.1,Chrome Mobile iOS 50.0.2661,M...",60,19,32,0.533333
14770,"mobile,iOS 11.2.6,Firefox 20.0.0.1618,Mozilla/...",63,14,33,0.523810


# Derive KG-based rarity & risk features and merge into df_kg.

Compute rarity in global stats tables

In [13]:
import numpy as np
import pandas as pd

#  Make copies so we don't accidentally overwrite the original stats
loc_stats = location_global_stats.copy()
dev_stats = device_global_stats.copy()
asn_stats = asn_global_stats.copy()

# Helper: rarity from events → higher = rarer
def compute_rarity_from_events(events_series):
    rarity_raw = 1.0 / (events_series.astype(float) + 1.0)
    # normalize to [0,1]
    return (rarity_raw - rarity_raw.min()) / (rarity_raw.max() - rarity_raw.min() + 1e-9)

# Location rarity (KG-based)
loc_stats["loc_rarity_kg"] = compute_rarity_from_events(loc_stats["events"])

# Device rarity (KG-based)
dev_stats["dev_rarity_kg"] = compute_rarity_from_events(dev_stats["events"])

# ASN rarity (KG-based)
asn_stats["asn_rarity_kg"] = compute_rarity_from_events(asn_stats["events"])

Interpretation:


```
events = how many login events used that location_id in the whole dataset.
	•	loc_rarity_kg = rarity score between 0 and 1
	•	More events → more common → lower rarity
	•	Fewer events → rarer → higher rarity
```




In [14]:
loc_stats[["events", "loc_rarity_kg"]].head()


,events,loc_rarity_kg
0,10,0.181788
1,1,1.000000
2,9,0.199970
3,1,1.000000
4,6,0.285688


	•	Locations with only 1 event get rarity 1.0 (rarest end).
	•	Locations with 6, 9, 10 events have lower rarity (~0.18–0.29), meaning:
	•	they are not super common, but clearly more common than “only seen once” locations.

In [15]:
asn_stats[["events", "asn_rarity_kg"]].head()

,events,asn_rarity_kg
0,6,0.285698
1,6,0.285698
2,44,0.044422
3,5,0.333318
4,2,0.666659


	•	ASN with 2 events → rarity ~0.67 (quite rare)
	•	ASN with 5–6 events → rarity ~0.28–0.33
	•	ASN with 44 events → rarity ~0.044 (much more common)



Quantile buckets from rarity





```
gives you discrete categories 0–3 (0 = most common, 3 = rarest).
These are good both for analysis and as model features.
```





In [16]:
# 4 quantiles: 0 (most common) → 3 (rarest)
loc_stats["loc_rarity_kg_q"] = pd.qcut(
    loc_stats["loc_rarity_kg"],
    q=4,
    labels=False,
    duplicates="drop"
).astype(int)

dev_stats["dev_rarity_kg_q"] = pd.qcut(
    dev_stats["dev_rarity_kg"],
    q=4,
    labels=False,
    duplicates="drop"
).astype(int)

asn_stats["asn_rarity_kg_q"] = pd.qcut(
    asn_stats["asn_rarity_kg"],
    q=4,
    labels=False,
    duplicates="drop"
).astype(int)

loc_stats[["loc_rarity_kg", "loc_rarity_kg_q"]].head()

,loc_rarity_kg,loc_rarity_kg_q
0,0.181788,0
1,1.000000,2
2,0.199970,1
3,1.000000,2
4,0.285688,1


You now have a discrete version of rarity:

	•	0 = very common
	•	1 = somewhat common
	•	2 = somewhat rare
	•	3 = rarest

Interpretation:



```
splitting rarity into buckets 0–3 (0 = lowest rarity, 3 = highest rarity):
```

	•	A location with rarity 0.1817 got bucket 0 → relatively common among all locations.
	•	A location with rarity 1.0 got bucket 2 in your sample — this just means:
	•	After quantile splitting across all 5,841 locations, that particular value fell into the 3rd quartile (label 2).
	•	(Because of ties / distribution shape, not all “1.0” values are necessarily in the very top bucket if duplicates="drop" triggered.)

Don’t overthink the exact bucket id; the important thing is:

	•	You now have a discrete version of rarity:
	•	0 = very common
	•	1 = somewhat common
	•	2 = somewhat rare
	•	3 = rarest


# Merge KG rarity & attack-rate back into df_kg

In [17]:
# First, rename attack_rate columns so they are explicit
loc_stats = loc_stats.rename(columns={"attack_rate": "loc_attack_rate_kg"})
dev_stats = dev_stats.rename(columns={"attack_rate": "dev_attack_rate_kg"})
asn_stats = asn_stats.rename(columns={"attack_rate": "asn_attack_rate_kg"})

# Merge location KG features
df_kg = df_kg.merge(
    loc_stats[["location_id", "loc_rarity_kg", "loc_rarity_kg_q", "loc_attack_rate_kg"]],
    on="location_id",
    how="left"
)

# Merge device KG features
df_kg = df_kg.merge(
    dev_stats[["device_id", "dev_rarity_kg", "dev_rarity_kg_q", "dev_attack_rate_kg"]],
    on="device_id",
    how="left"
)

# Merge ASN KG features
df_kg = df_kg.merge(
    asn_stats[["asn_id", "asn_rarity_kg", "asn_rarity_kg_q", "asn_attack_rate_kg"]],
    on="asn_id",
    how="left"
)

# Quick look at the new columns
df_kg[
    [
        "location_id", "loc_rarity_kg", "loc_rarity_kg_q", "loc_attack_rate_kg",
        "device_id", "dev_rarity_kg", "dev_rarity_kg_q", "dev_attack_rate_kg",
        "asn_id", "asn_rarity_kg", "asn_rarity_kg_q", "asn_attack_rate_kg"
    ]
].head()

,location_id,loc_rarity_kg,loc_rarity_kg_q,loc_attack_rate_kg,device_id,dev_rarity_kg,dev_rarity_kg_q,dev_attack_rate_kg,asn_id,asn_rarity_kg,asn_rarity_kg_q,asn_attack_rate_kg
0,"no,oslo county,oslo",0.000042,0,0.013573,"desktop,Mac OS X 10.14.6,Chrome 69.0.3497.17.1...",0.000021,0,0.023116,41164,0.000110,0,0.004911
1,"no,-,-",0.000021,0,0.012303,"mobile,iOS 7.1,Android 2.3.3.2672,Mozilla/5.0 ...",0.000000,0,0.014447,49310,0.054032,0,0.000000
2,"no,-,-",0.000021,0,0.012303,"mobile,iOS 7.1,Android 2.3.3.2672,Mozilla/5.0 ...",0.000000,0,0.014447,49310,0.054032,0,0.000000
3,"no,vestfold og telemark,holmestrand",0.010013,0,0.000000,"mobile,iOS 13.4,Chrome Mobile 81.0.4044.2033,M...",0.499968,1,0.000000,29695,0.000000,0,0.006373
4,"no,-,-",0.000021,0,0.012303,"mobile,Android 13.0,Opera Mobile 52.1.2254,Moz...",0.000296,0,0.026959,29695,0.000000,0,0.006373


In [18]:
df_kg[[
    "loc_rarity_kg", "loc_attack_rate_kg",
    "dev_rarity_kg", "dev_attack_rate_kg",
    "asn_rarity_kg", "asn_attack_rate_kg"
]].describe()

,loc_rarity_kg,loc_attack_rate_kg,dev_rarity_kg,dev_attack_rate_kg,asn_rarity_kg,asn_attack_rate_kg
count,300000.000000,300000.000000,300000.000000,300000.000000,300000.000000,300000.000000
mean,0.028526,0.092173,0.094343,0.092173,0.013580,0.092173
std,0.108623,0.158851,0.202402,0.184336,0.070730,0.164200
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,0.000021,0.000000,0.000685,0.008448,0.000000,0.004911
50%,0.000665,0.012303,0.008034,0.018582,0.000110,0.006373
75%,0.006272,0.105706,0.071369,0.053398,0.001008,0.104741
max,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000


a) Rarity columns (*_rarity_kg)

	•	Means are very small (0.0285, 0.0943, 0.0136):

Most events use fairly common locations/devices/ASNs.

	•	75% quantile values are still very small:
	•	For locations, 75% of events have loc_rarity_kg ≤ 0.00627.
	•	For ASNs, 75% have asn_rarity_kg ≤ 0.0010.

👉 That means truly rare infrastructure is rare — exactly as expected.

b) Attack rate columns (*_attack_rate_kg)

	•	The mean = 0.092173 for all three → that matches your global attack rate ~9.2%.

This is because for each event, you’re just attaching the attack rate of its location/device/ASN.

	•	Median (50%) location_attack_rate ~ 0.0123:
	•	For 50% of events, the associated location has attack rate ~1.2% or less.
	•	75% quantile location_attack_rate ~ 0.1057:
	•	Only 25% of events are associated with locations having attack rate ≥ ~10.6%.

Similarly for ASNs and devices.

👉 Interpretation:

	•	Most events are associated with locations/devices/ASNs that historically look pretty safe.
	•	A smaller subset of events are tied to entities with much higher observed attack rates, and these will be very useful as risk signals in your intent model.

Result:

From the knowledge graph, we compute global statistics for each location, device, and ASN, including total events, unique users, and observed attack rate. We then derive KG-based rarity scores (loc_rarity_kg, dev_rarity_kg, asn_rarity_kg) and corresponding quantile buckets, and join these back to each login event. This provides a population-level context for each authentication event, indicating whether it involves common benign infrastructure or rare, high-risk infrastructure.



# KG-based features


## Entity rarity & risk (global KG stats per event) population-level behavior

	•	Location-based KG features
	•	loc_rarity_kg → normalized rarity of this location_id (0 = very common, 1 = very rare)
	•	loc_rarity_kg_q → rarity bucket (0–3)
	•	loc_attack_rate_kg → historical attack rate for this location
	•	Device-based KG features
	•	dev_rarity_kg → rarity of this device_id
	•	dev_rarity_kg_q → bucket
	•	dev_attack_rate_kg → historical attack rate for this device fingerprint
	•	ASN-based KG features
	•	asn_rarity_kg → rarity of this asn_id
	•	asn_rarity_kg_q → bucket
	•	asn_attack_rate_kg → historical attack rate for this ASN

## User–entity relationship features (per-user KG edges)

These come from comparing current event to user history (which is effectively your user–entity edges):

	•	Novelty / change:
	•	is_new_device_for_user (you derived earlier)
	•	is_new_ip_for_user
	•	new_location_flag (user appears from a new location)
	•	new_asn_flag
	•	new_device_flag
	•	device_change_rate (how often the user changes devices)


## Time-behavior & regularity (using KG-like historical memory)

Not strictly “graph edges”, but they’re computed from the same historical behavior of each user:

	•	is_off_hours
	•	is_unusual_time_for_user
	•	user_offhour_rate
	•	user_hour_std, user_hour_std_q

These capture how this login’s time compares to the user’s normal time distribution, which is stored via historical events


# Bulding KG v2

# device_user_count, asn_user_count (from global stats)

In [19]:
# Make copies to be safe
dev_stats = device_global_stats.copy()
asn_stats = asn_global_stats.copy()

# Assume these have columns: device_id / asn_id and 'users'
# If your key columns are called differently, rename them here.

dev_stats = dev_stats.rename(columns={"users": "device_user_count"})
asn_stats = asn_stats.rename(columns={"users": "asn_user_count"})

# Merge into df_kg
df_kg = df_kg.merge(
    dev_stats[["device_id", "device_user_count"]],
    on="device_id",
    how="left"
)

df_kg = df_kg.merge(
    asn_stats[["asn_id", "asn_user_count"]],
    on="asn_id",
    how="left"
)

# Fill any missing counts with 1 (seen only once)
df_kg["device_user_count"] = df_kg["device_user_count"].fillna(1).astype(int)
df_kg["asn_user_count"] = df_kg["asn_user_count"].fillna(1).astype(int)

df_kg[["device_id", "device_user_count", "asn_id", "asn_user_count"]].head()

,device_id,device_user_count,asn_id,asn_user_count
0,"desktop,Mac OS X 10.14.6,Chrome 69.0.3497.17.1...",14081,41164,8113
1,"mobile,iOS 7.1,Android 2.3.3.2672,Mozilla/5.0 ...",17181,49310,20
2,"mobile,iOS 7.1,Android 2.3.3.2672,Mozilla/5.0 ...",17181,49310,20
3,"mobile,iOS 13.4,Chrome Mobile 81.0.4044.2033,M...",2,29695,44988
4,"mobile,Android 13.0,Opera Mobile 52.1.2254,Moz...",3033,29695,44988


	There are devices used by thousands of different users (14k+, 17k+).
	•	In a synthetic dataset, that likely represents “shared infrastructure”: proxy / emulator / bot farm device signatures.
	•	device_attack_user_count shows how many distinct users on that device had at least one attack.
	•	E.g., 313 different users on the first device had attack-labelled events.

So shared_device_flag = 1 means:

“This device is heavily shared and has at least one attacking user on it.”

Shared infra risk features

device_attack_user_count and asn_attack_user_count

In [20]:
# Helper: per (device, user) whether this user ever attacked on this device
dev_user_attack = (
    df_kg
    .groupby(["device_id", "User ID"], as_index=False)["Is Attack IP"]
    .max()
)

dev_attack_users = (
    dev_user_attack
    .groupby("device_id")["Is Attack IP"]
    .sum()
    .reset_index()
    .rename(columns={"Is Attack IP": "device_attack_user_count"})
)

# Same for ASN
asn_user_attack = (
    df_kg
    .groupby(["asn_id", "User ID"], as_index=False)["Is Attack IP"]
    .max()
)

asn_attack_users = (
    asn_user_attack
    .groupby("asn_id")["Is Attack IP"]
    .sum()
    .reset_index()
    .rename(columns={"Is Attack IP": "asn_attack_user_count"})
)

# Merge into df_kg
df_kg = df_kg.merge(dev_attack_users, on="device_id", how="left")
df_kg = df_kg.merge(asn_attack_users, on="asn_id", how="left")

df_kg["device_attack_user_count"] = df_kg["device_attack_user_count"].fillna(0).astype(int)
df_kg["asn_attack_user_count"] = df_kg["asn_attack_user_count"].fillna(0).astype(int)

df_kg[["device_id", "device_user_count", "device_attack_user_count"]].head()

,device_id,device_user_count,device_attack_user_count
0,"desktop,Mac OS X 10.14.6,Chrome 69.0.3497.17.1...",14081,313
1,"mobile,iOS 7.1,Android 2.3.3.2672,Mozilla/5.0 ...",17181,221
2,"mobile,iOS 7.1,Android 2.3.3.2672,Mozilla/5.0 ...",17181,221
3,"mobile,iOS 13.4,Chrome Mobile 81.0.4044.2033,M...",2,0
4,"mobile,Android 13.0,Opera Mobile 52.1.2254,Moz...",3033,69


shared_device_flag and shared_asn_flag

In [21]:
# “shared” = used by ≥ 5 users, and
	# •	at least 1 of those users has had an attack on this entity.

SHARED_USER_THRESHOLD = 5

df_kg["shared_device_flag"] = (
    (df_kg["device_user_count"] >= SHARED_USER_THRESHOLD) &
    (df_kg["device_attack_user_count"] >= 1)
).astype(int)

df_kg["shared_asn_flag"] = (
    (df_kg["asn_user_count"] >= SHARED_USER_THRESHOLD) &
    (df_kg["asn_attack_user_count"] >= 1)
).astype(int)

df_kg[["device_id", "device_user_count", "device_attack_user_count", "shared_device_flag"]].head()

,device_id,device_user_count,device_attack_user_count,shared_device_flag
0,"desktop,Mac OS X 10.14.6,Chrome 69.0.3497.17.1...",14081,313,1
1,"mobile,iOS 7.1,Android 2.3.3.2672,Mozilla/5.0 ...",17181,221,1
2,"mobile,iOS 7.1,Android 2.3.3.2672,Mozilla/5.0 ...",17181,221,1
3,"mobile,iOS 13.4,Chrome Mobile 81.0.4044.2033,M...",2,0,0
4,"mobile,Android 13.0,Opera Mobile 52.1.2254,Moz...",3033,69,1


In [22]:
print("Attack rate by shared_device_flag:")
print(df_kg.groupby("shared_device_flag")["Is Attack IP"].mean())

print("\nAttack rate by shared_asn_flag:")
print(df_kg.groupby("shared_asn_flag")["Is Attack IP"].mean())

Attack rate by shared_device_flag:
shared_device_flag
0    0.128578
1    0.077274
Name: Is Attack IP, dtype: float64

Attack rate by shared_asn_flag:
shared_asn_flag
0    0.017670
1    0.106461
Name: Is Attack IP, dtype: float64


This is interesting:
	•	You’d expect shared malicious devices to be riskier.
	•	But in this dataset, shared devices actually have lower attack rate than non-shared ones.

Possible reasons (and exactly how you can explain it):
	•	A lot of attack traffic may be coming from less-shared, “dedicated” devices.
	•	The synthetic generator might have made “attack devices” not necessarily the most shared ones.
	•	Shared devices might be representing things like common mobile/browser signatures that many benign users also use.

This is much more aligned with intuition:
	•	When shared_asn_flag = 1:
	•	ASN is used by many users, and
	•	There is at least one attacking user on that ASN.
	•	That group has an attack rate 6× higher than the non-shared-ASN group (10.6% vs 1.8%).

👉 This is a strong infra signal:

“If you’re on an ASN that’s widely used and has known attack users, your login is considerably more risky.”

# User diversity features

 Compute per-user counts

In [23]:
user_div = (
    df_kg
    .groupby("User ID")
    .agg(
        user_device_count=("device_id", "nunique"),
        user_location_count=("location_id", "nunique"),
        user_asn_count=("asn_id", "nunique"),
    )
    .reset_index()
)

df_kg = df_kg.merge(user_div, on="User ID", how="left")

df_kg[["User ID", "user_device_count", "user_location_count", "user_asn_count"]].head()

,User ID,user_device_count,user_location_count,user_asn_count
0,-9223287066183308537,1,1,1
1,-9223258649185196422,1,1,1
2,-9223258649185196422,1,1,1
3,-9223200578825105501,1,1,1
4,-9223199305075633823,1,1,1


We Found:

	•	user_device_count → how many different devices this user has ever used
	•	user_location_count → how many different locations
	•	user_asn_count → how many different ASNs


  “Most users authenticate from a small number of devices, locations, and ASNs, while a small group exhibits high diversity. These diversity features help distinguish normal multi-device usage from potentially compromised accounts operating over many networks or geographies"

👉 How to use this:
	•	As a feature, not as a rule:
	•	The model can learn: “extremely high diversity + other signals = riskier”.
	•	As a story point in your report:
“We observe that while most users authenticate from a single device and location, a small population shows very high infrastructure diversity. Our user diversity features (user_device_count, user_location_count, user_asn_count) capture these patterns and allow the intent model to distinguish stable accounts from those operating across unusually many devices and networks.”

In [24]:
df_kg[["user_device_count", "user_location_count", "user_asn_count"]].describe()

,user_device_count,user_location_count,user_asn_count
count,300000.000000,300000.000000,300000.000000
mean,5094.421980,1546.854973,746.549467
std,6890.702112,2091.137895,1008.521730
min,1.000000,1.000000,1.000000
25%,1.000000,1.000000,1.000000
50%,1.000000,1.000000,1.000000
75%,14417.000000,4376.000000,2111.000000
max,14417.000000,4376.000000,2111.000000


Temporal infra risk (*_recent_attack_rate_7d)

In [26]:
import pandas as pd

# Ensure sorted by time and reset index to have a clean integer-based index
df_kg = df_kg.sort_values("Login Timestamp").reset_index(drop=True)

# Initialize the new column with a default value
df_kg["asn_recent_attack_rate_7d"] = 0.0

# Group by ASN
# .groups.items() returns (asn_value, list_of_original_df_kg_indices)
for asn, grp_original_indices in df_kg.groupby("asn_id").groups.items():
    # Select the subset of df_kg for the current ASN using its original df_kg indices
    # .copy() is used to prevent SettingWithCopyWarning
    grp_data = df_kg.loc[grp_original_indices, ["Login Timestamp", "Is Attack IP"]].copy()

    # Ensure this subset is sorted by timestamp for correct rolling window calculation
    grp_data = grp_data.sort_values("Login Timestamp")

    # The index of `grp_data` at this point still consists of the original integer indices from df_kg,
    # but they might be reordered due to the `sort_values` call. This is crucial for assignment.
    current_df_kg_indices_for_group = grp_data.index

    # Temporarily set 'Login Timestamp' as the index for rolling calculation
    grp_data_indexed = grp_data.set_index("Login Timestamp")

    # Calculate the rolling mean for 'Is Attack IP'
    recent_rate_series = grp_data_indexed["Is Attack IP"].rolling("7D", min_periods=1).mean()

    # Assign the calculated rates back to the main df_kg DataFrame.
    # The order of `recent_rate_series.values` matches the sorted order of `grp_data_indexed`,
    # which in turn matches the order of `current_df_kg_indices_for_group` (the reordered integer indices).
    df_kg.loc[current_df_kg_indices_for_group, "asn_recent_attack_rate_7d"] = recent_rate_series.values

# Display descriptive statistics for the newly created feature
df_kg["asn_recent_attack_rate_7d"].describe()

,asn_recent_attack_rate_7d
count,300000.000000
mean,0.093043
std,0.167153
min,0.000000
25%,0.004822
50%,0.006668
75%,0.082540
max,1.000000


Interpretation:
	•	Global mean ≈ 0.093 matches your global attack rate (~9.2%) → good sanity check.
	•	Most ASNs are quiet recently:
	•	25% of events see ASN recent rate ≤ 0.0048
	•	50% ≤ 0.0067
→ many ASNs have very low recent attack activity.
	•	Top quartile (75%) is at 0.0825 → some ASNs have ~8–10% recent attack rates.
	•	max = 1.0 → for some ASNs, in a 7-day window, every recent event was an attack (super “hot” ASN).

👉 This is a very realistic & powerful feature:

“Even if an ASN wasn’t historically bad over the whole dataset, if in the last 7 days it’s been used mostly for attacks, we treat logins from this ASN as significantly higher risk.”